# Building Energy Performance Prediction: Supporting Sustainable Urbanization

## Problem Statement

Buildings are responsible for approximately **40% of global energy consumption** and **36% of CO2 emissions**, with heating and cooling accounting for the majority of this usage. As urbanization accelerates—particularly in developing nations where 75% of buildings expected to exist by 2030 have yet to be constructed—there is a critical need to design energy-efficient buildings from the outset.

Traditional building energy assessment occurs post-construction, when design changes are costly or impossible. **This project addresses the question:**

> **Can we accurately predict a building's heating and cooling energy requirements based solely on its design parameters before construction begins?**

By developing a machine learning model that predicts energy loads from architectural features (building shape, surface area, window coverage, orientation, etc.), this project aims to:

1. **Enable data-driven architectural decisions** that prioritize energy efficiency
2. **Reduce carbon emissions** from the building sector
3. **Support sustainable urban development** by identifying key design parameters that minimize energy consumption
4. **Demonstrate ML applications** in climate tech and sustainable design

## Dataset Overview

### Source
- **Dataset:** Energy Efficiency Dataset
- **Repository:** UCI Machine Learning Repository  
- **Authors:** Tsanas, A. & Xifara, A. (2012)
- **Publication:** "Accurate quantitative estimation of energy performance of residential buildings using statistical machine learning tools" - Energy and Buildings, Vol. 49, pp. 560-567
- **DOI:** 10.1016/j.enbuild.2012.03.003
- **Total Samples:** 768 simulated buildings
- **Features:** 8 input variables, 2 target variables
- **Missing Values:** None

### About the Dataset

This dataset was created using **Ecotect**, a professional building energy simulation software, to model 768 different residential building designs. The simulations represent buildings located in **Athens, Greece**, with a Mediterranean climate characterized by hot summers and mild winters.

**Key Characteristics:**
- All buildings share the same **volume** (771.75 m³) to ensure controlled comparison
- Buildings vary in **shape, window coverage, orientation**, and other architectural parameters
- Constructed using modern building materials with low heat loss coefficients (U-values)
- Data represents **high-fidelity simulations** validated against real-world building performance

The dataset enables prediction of heating and cooling energy requirements based solely on design parameters, supporting energy-efficient architectural decisions **before construction** begins.

---

### Features Description

#### **Input Variables (X1 - X8): Building Design Parameters**

| Feature | Description | Unit | Range/Values | Impact on Energy |
|---------|-------------|------|--------------|------------------|
| **X1 - Relative Compactness (RC)** | Ratio measuring how compact the building shape is. Higher values indicate more cube-like structures; lower values indicate spread-out, elongated shapes. | Dimensionless | 0.62 - 0.98 | Compact buildings (high RC) have less surface area for heat exchange → **lower energy needs** |
| **X2 - Surface Area (SA)** | Total exterior surface area of the building (walls + roof + floor) | m² | 514 - 808 | Larger surface area → **more heat loss/gain** → higher energy needs |
| **X3 - Wall Area (WA)** | Total area of exterior walls only | m² | 245 - 416 | More wall area → more heat exchange → **higher energy needs** |
| **X4 - Roof Area (RA)** | Total roof surface area | m² | 110 - 220 | Larger roof → more solar heat gain → **higher cooling load** |
| **X5 - Overall Height (OH)** | Vertical height of the building | meters | 3.5 or 7.0 | Taller buildings affect air circulation and heating distribution |
| **X6 - Orientation (O)** | Cardinal direction the building faces | Categorical | 2=North, 3=East, 4=South, 5=West | Affects solar exposure (South faces get most sun in Northern Hemisphere) |
| **X7 - Glazing Area (GA)** | Windows as percentage of floor area | % | 0, 10, 25, 40 | **Most critical feature:** More windows → more heat loss/gain → **significantly higher energy needs** |
| **X8 - Glazing Area Distribution (GAD)** | How windows are distributed across building faces | Categorical | 0-5 (see below) | Affects which sides receive solar heat |

**Glazing Area Distribution (GAD) Values:**
- **0:** No glazing (no windows)
- **1:** Uniform (25% on each side: N, S, E, W)
- **2:** North-heavy (55% North, 15% each other side)
- **3:** East-heavy (55% East, 15% each other side)
- **4:** South-heavy (55% South, 15% each other side)
- **5:** West-heavy (55% West, 15% each other side)

---

#### **Target Variables (Y1 - Y2): Energy Requirements**

| Target | Description | Unit | Range | Purpose |
|--------|-------------|------|-------|---------|
| **Y1 - Heating Load (HL)** | Energy required to maintain comfortable indoor temperature during cold weather | kWh | 6 - 43 | Predict heating costs and winter energy consumption |
| **Y2 - Cooling Load (CL)** | Energy required to maintain comfortable indoor temperature during hot weather | kWh | 10 - 48 | Predict cooling costs and summer energy consumption |

**Lower values** indicate more energy-efficient buildings that require less heating/cooling.

---

### Key Insights from Prior Research

Studies analyzing this dataset (Tsanas & Xifara, 2012; subsequent research using fuzzy logic and neural networks) have revealed:

1. **Most Influential Features:**
   - **Glazing Area (GA)** is the single most important predictor of both heating and cooling loads
   - **Relative Compactness (RC)** is the second most important feature
   - These two features alone can achieve prediction accuracy comparable to using all 8 features

2. **Feature Redundancy:**
   - Surface Area (SA), Wall Area (WA), and Roof Area (RA) are mathematically correlated with Relative Compactness (RC)
   - Orientation (O) showed surprisingly minimal correlation with energy loads in this dataset

3. **Design Trade-offs:**
   - Windows provide natural light but create thermal weak points
   - Compact building shapes are more energy-efficient than spread-out designs
   - South-facing windows (in Northern Hemisphere) maximize passive solar heating in winter but increase cooling needs in summer

---

### Project Focus

For this project, we will:
1. **Primary objective:** Build a linear regression model to predict **Heating Load (Y1)**
2. **Secondary analysis:** Explore feature importance and correlations
3. **Validation:** Compare model performance with baseline results from published research
4. **Feature engineering:** Investigate whether using only the 2 most important features (RC + GA) maintains prediction accuracy

This analysis will demonstrate how machine learning can support sustainable building design by identifying optimal architectural parameters for energy efficiency.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
# The first few rows of the data 
data = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Project_Linear_Regression\Linear_Regression\data_energy.csv')
data.head()

,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


### Renaming Columns for Clarity

The dataset uses generic column names (X1-X8 for features, Y1-Y2 for targets). 
We'll rename them to descriptive names based on the dataset documentation.

In [19]:
new_column_names = ['relative_compactness','surface_area','wall_area','roof_area','overall_height','orientation','glazing_area',
                    'glazing_area_distribution','heating_load','cooling_load']

data.columns = new_column_names

In [22]:
data.head(10)

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_area_distribution,heating_load,cooling_load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28
5,0.90,563.5,318.5,122.50,7.0,3,0.0,0,21.46,25.38
6,0.90,563.5,318.5,122.50,7.0,4,0.0,0,20.71,25.16
7,0.90,563.5,318.5,122.50,7.0,5,0.0,0,19.68,29.60
8,0.86,588.0,294.0,147.00,7.0,2,0.0,0,19.50,27.30
9,0.86,588.0,294.0,147.00,7.0,3,0.0,0,19.95,21.97


In [20]:
# Information of the data
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   relative_compactness       768 non-null    float64
 1   surface_area               768 non-null    float64
 2   wall_area                  768 non-null    float64
 3   roof_area                  768 non-null    float64
 4   overall_height             768 non-null    float64
 5   orientation                768 non-null    int64  
 6   glazing_area               768 non-null    float64
 7   glazing_area_distribution  768 non-null    int64  
 8   heating_load               768 non-null    float64
 9   cooling_load               768 non-null    float64
dtypes: float64(8), int64(2)
memory usage: 60.1 KB


In [17]:
# Basic Descriptive statistics--
data.describe()

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_area_distribution,heating_load,cooling_load
count,768.000000,768.000000,768.000000,768.000000,768.00000,768.000000,768.000000,768.00000,768.000000,768.000000
mean,0.764167,671.708333,318.500000,176.604167,5.25000,3.500000,0.234375,2.81250,22.307201,24.587760
std,0.105777,88.086116,43.626481,45.165950,1.75114,1.118763,0.133221,1.55096,10.090196,9.513306
min,0.620000,514.500000,245.000000,110.250000,3.50000,2.000000,0.000000,0.00000,6.010000,10.900000
25%,0.682500,606.375000,294.000000,140.875000,3.50000,2.750000,0.100000,1.75000,12.992500,15.620000
50%,0.750000,673.750000,318.500000,183.750000,5.25000,3.500000,0.250000,3.00000,18.950000,22.080000
75%,0.830000,741.125000,343.000000,220.500000,7.00000,4.250000,0.400000,4.00000,31.667500,33.132500
max,0.980000,808.500000,416.500000,220.500000,7.00000,5.000000,0.400000,5.00000,43.100000,48.030000
